# 16 高级特征筛选 + 超参数调优

覆盖 NullImportanceSelector / BorutaSelector / RFESelector / SequentialFeatureSelector /
StepwiseSelector / Chi2Selector / FTestSelector / MutualInfoSelector + ModelTuner。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from hscredit import (germancredit, init_setting, seed_everything,
    NullImportanceSelector, RFESelector, SequentialFeatureSelector,
    MutualInfoSelector, Chi2Selector, FTestSelector,
    CompositeFeatureSelector, SelectionReportCollector,
)
seed_everything(42); init_setting()
df = germancredit()
target = 'creditability'
num_feats = df.select_dtypes('number').columns.drop(target).tolist()
X = df[num_feats]; y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
print('特征数:', len(num_feats))

## 1. NullImportanceSelector — 零重要性特征剔除

In [ ]:
ni_sel = NullImportanceSelector(
    n_iterations=20, random_state=42,
    importance_threshold_percentile=25
)
ni_sel.fit(X_tr, y_tr)
X_ni = ni_sel.transform(X_tr)
print(f'NullImportance: {X_tr.shape[1]} → {X_ni.shape[1]} 个特征')
print('保留:', X_ni.columns.tolist())

## 2. RFESelector — 递归特征消除

In [ ]:
from sklearn.linear_model import LogisticRegression as SKLR
rfe_sel = RFESelector(estimator=SKLR(max_iter=1000), n_features_to_select=5, step=1)
rfe_sel.fit(X_tr, y_tr)
X_rfe = rfe_sel.transform(X_tr)
print(f'RFE: {X_tr.shape[1]} → {X_rfe.shape[1]} 个特征')
print('保留:', X_rfe.columns.tolist())

## 3. SequentialFeatureSelector — 逐步添加/删除

In [ ]:
sfs_sel = SequentialFeatureSelector(
    estimator=SKLR(max_iter=1000),
    n_features_to_select=5,
    direction='forward',
    scoring='roc_auc', cv=3,
)
sfs_sel.fit(X_tr, y_tr)
X_sfs = sfs_sel.transform(X_tr)
print(f'SFS forward: 保留 {X_sfs.shape[1]} 个')
print('保留:', X_sfs.columns.tolist())

## 4. Chi2 / FTest / MutualInfo 筛选器

In [ ]:
chi2_sel = Chi2Selector(k=6)
chi2_sel.fit(X_tr.abs(), y_tr)
print('Chi2保留:', chi2_sel.transform(X_tr.abs()).columns.tolist())

In [ ]:
ft_sel = FTestSelector(k=6)
ft_sel.fit(X_tr, y_tr)
print('FTest保留:', ft_sel.transform(X_tr).columns.tolist())

In [ ]:
mi_sel = MutualInfoSelector(k=6, random_state=42)
mi_sel.fit(X_tr, y_tr)
print('MutualInfo保留:', mi_sel.transform(X_tr).columns.tolist())

## 5. BorutaSelector — Boruta全相关特征筛选

In [ ]:
try:
    from hscredit import BorutaSelector
    boruta_sel = BorutaSelector(n_estimators=50, random_state=42, max_iter=30)
    boruta_sel.fit(X_tr, y_tr)
    X_boruta = boruta_sel.transform(X_tr)
    print(f'Boruta: {X_tr.shape[1]} → {X_boruta.shape[1]} 个')
    print('确认相关:', boruta_sel.support_.sum(), '个')
except Exception as e:
    print(f'Boruta: {e}')

## 6. StepwiseSelector — 逐步回归筛选

In [ ]:
from hscredit import CompositeFeatureSelector
from hscredit.core.selectors.stepwise_selector import StepwiseSelector
step_sel = StepwiseSelector(
    direction='both', scoring='roc_auc', cv=3,
    threshold_in=0.01, threshold_out=0.05,
)
step_sel.fit(X_tr, y_tr)
X_step = step_sel.transform(X_tr)
print(f'Stepwise: {X_tr.shape[1]} → {X_step.shape[1]} 个')
print('保留:', X_step.columns.tolist())

## 7. RulesClassifier — 规则集分类模型

In [ ]:
from hscredit.core.models.rule_classifier import RulesClassifier
rules_config = [
    {'id': 'r1', 'name': '长期高额', 'expr': '`duration.in.month` > 36 and `credit.amount` > 5000'},
    {'id': 'r2', 'name': '年轻借款', 'expr': '`age.in.years` < 25'},
    {'id': 'r3', 'name': '高额贷款', 'expr': '`credit.amount` > 8000'},
]
clf = RulesClassifier(rules=rules_config, logic='or', default_label=0)
clf.fit(df[num_feats], df[target])
preds = clf.predict(df[num_feats])
hit_rate = preds.mean()
print(f'规则命中率: {hit_rate:.2%}')
print('各规则命中详情:')
for r in clf.rule_results_:
    print(f'  {r.rule_name}: {r.matched_count}/{len(df)} ({r.matched_count/len(df):.1%})')

## 8. ModelTuner — 超参数调优（Optuna）

In [ ]:
try:
    from hscredit import ModelTuner
    from hscredit import RandomForestRiskModel
    search_space = {
        'n_estimators': {'type': 'int', 'low': 50, 'high': 200},
        'max_depth': {'type': 'int', 'low': 3, 'high': 8},
        'min_samples_leaf': {'type': 'int', 'low': 10, 'high': 50},
    }
    tuner = ModelTuner(
        model_class=RandomForestRiskModel,
        search_space=search_space,
        metric='ks',
        direction='maximize',
        cv=3,
    )
    best_params = tuner.fit(X_tr, y_tr, n_trials=20)
    print('最优参数:', best_params)
    print('最优KS:', tuner.best_value_)
except ImportError:
    print('optuna未安装，跳过调优')

## 9. AutoTuner — 自动调优

In [ ]:
try:
    from hscredit import AutoTuner, TuningObjective
    auto = AutoTuner(objective=TuningObjective.KS, n_trials=20, cv=3)
    auto.fit(X_tr, y_tr)
    print('AutoTuner最优模型:', auto.best_model_)
    print('最优KS:', auto.best_score_)
    proba_auto = auto.predict_proba(X_te)
    from hscredit.core.metrics import ks, auc
    print(f'测试集 KS:{ks(y_te,proba_auto):.4f}  AUC:{auc(y_te,proba_auto):.4f}')
except (ImportError, Exception) as e:
    print(f'AutoTuner: {e}')